# NB01 – Data Collection

## European Air Quality and Weather

### Research question

How does air quality vary across major European cities, and which weather conditions are associated with worse pollution?

### Purpose

This notebook collects air-quality and weather data for a selection of major European cities. The data is collected using the OpenWeather API.

The original API responses are saved in the `data/raw` folder so that the collection process is reproducible and the source data remains unchanged.

The collected variables include:

- Air Quality Index
- PM2.5
- PM10
- Nitrogen dioxide
- Ozone
- Carbon monoxide
- Temperature
- Humidity
- Wind speed
- Atmospheric pressure

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

## Create Project Paths

In [3]:
# The notebook is stored inside the notebooks folder,
# so ".." refers to the main final-project folder.
PROJECT_DIR = Path("..")
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

# Create the folders if they do not already exist.
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR.resolve())
print("Raw data folder:", RAW_DATA_DIR.resolve())

Project folder: /files/assignments/final-project
Raw data folder: /files/assignments/final-project/data/raw


## Load the API key

In [4]:
# Load variables stored in the .env file.
load_dotenv(PROJECT_DIR / ".env")

API_KEY = os.getenv("OPENWEATHER_API_KEY")

if API_KEY is None:
    raise ValueError(
        "The OpenWeather API key was not found. "
        "Check that the .env file exists in the final-project folder."
    )

print("API key loaded successfully.")

API key loaded successfully.


## Cities included in the project

The project compares ten major European cities from different parts of Europe. Using several cities allows the analysis to examine both variation in pollution and relationships between pollution and weather. The next thing I will be doing is defining the cities

In [5]:
cities = {
    "London": {"country": "GB", "latitude": 51.5074, "longitude": -0.1278},
    "Paris": {"country": "FR", "latitude": 48.8566, "longitude": 2.3522},
    "Berlin": {"country": "DE", "latitude": 52.5200, "longitude": 13.4050},
    "Madrid": {"country": "ES", "latitude": 40.4168, "longitude": -3.7038},
    "Rome": {"country": "IT", "latitude": 41.9028, "longitude": 12.4964},
    "Amsterdam": {"country": "NL", "latitude": 52.3676, "longitude": 4.9041},
    "Brussels": {"country": "BE", "latitude": 50.8503, "longitude": 4.3517},
    "Vienna": {"country": "AT", "latitude": 48.2082, "longitude": 16.3738},
    "Copenhagen": {"country": "DK", "latitude": 55.6761, "longitude": 12.5683},
    "Prague": {"country": "CZ", "latitude": 50.0755, "longitude": 14.4378},
}

cities_df = pd.DataFrame.from_dict(cities, orient="index")
cities_df.index.name = "city"
cities_df

,country,latitude,longitude
city,,,
London,GB,51.5074,-0.1278
Paris,FR,48.8566,2.3522
Berlin,DE,52.5200,13.4050
Madrid,ES,40.4168,-3.7038
Rome,IT,41.9028,12.4964
Amsterdam,NL,52.3676,4.9041
Brussels,BE,50.8503,4.3517
Vienna,AT,48.2082,16.3738
Copenhagen,DK,55.6761,12.5683
